# What each strategy would make of the dataset

A strategy says which instruments a window over a tile has to hold and how much
ground each of them has to reach. Change the demands and a different dataset comes
out the other end: more features but thinner coverage, or fewer features every one
of which was really looked at.

This notebook runs every strategy written in `src/survey/strategies/` over the
features computed on disk and reads the results side by side. It is a prediction of
the dataset, not the dataset: nothing here downloads or measures anything.

## How it is measured

**A feature is cut into tiles and every tile is searched on its own.** A tile earns a
window when the strategy's demands are all met inside it. A strategy admits a tile and
never a feature, so everything below is counted in tiles.

**Nothing is precomputed.** The search runs here, over the coverage artifacts, so a
strategy added or edited under `src/survey/strategies/` shows up on the next sweep
with no other step in between.

**A sweep is not free.** Every window a tile could hold is weighed, so the whole
catalogue takes a couple of hours across eight processes. `ONE_IN` below thins
the sweep to an even sample spread over the whole catalogue, rather than one that
stops at the first class. Set it to 1 for the real answer.

## Setup

In [ ]:
"""Import the sweep and the tables that read it."""

from visualization.dataset import progress
from visualization.dataset.stats import dataset
from visualization.dataset.tables import instruments, kept, windows

## Sweep the features

In [ ]:
"""Search every sampled feature under every strategy, then read the results."""

# Keep one feature in this many. Set to 1 to sweep the whole catalogue.
ONE_IN = 25

found = progress.swept(progress.sample(ONE_IN))
read = dataset.read(found)

## What each strategy asks

The demands a window has to meet all of, and how long a window may run. An
instrument listed as timeless is asked of the whole record rather than of the
window.

In [ ]:
"""Tabulate what every strategy asks of a window."""

kept.asked(read)

## What each strategy keeps

How many tiles were searched and how many of them were removed, how many tiles a
feature is cut into, and how much ground the kept tiles cover.

In [ ]:
"""Tabulate what every strategy would leave in the dataset."""

kept.plot(read)

## How much of a tile each instrument reaches

Averaged over every tile that earned a window, with the spread beside it. A tile an
instrument never reached counts as nothing rather than being left out, so a wide
spread means the instrument is on some tiles and absent from others.

The pixel column is the ground read in the instrument's own pixels, spread evenly
over the footprints that landed on the tile.

In [ ]:
"""Tabulate how much of a tile each instrument reaches, strategy by strategy."""

instruments.coverage(read)

## Where the instruments overlap

The ground the windows cover, split by which instruments are really on it. A cell is
counted once, under the instruments that reach it, so the rows do not overlap and
add up to the ground kept.

In [ ]:
"""Tabulate how much ground the instruments reach between them."""

instruments.overlap(read)

## The windows themselves

How long a window runs and how much of its tile it reaches, then what it keeps of the
observations it was offered. An observation is counted once per tile it was offered
to, since a footprint crossing two tiles is judged in each of them.

In [ ]:
"""Tabulate how long each strategy's windows run."""

windows.lengths(read)

In [ ]:
"""Tabulate what the windows keep of the observations they were offered."""

windows.taken(read)

## Which feature classes survive

Tiles kept of the tiles searched, class by class. A strategy that keeps only
craters is a different dataset from one that keeps craters and valles alike, however
close the totals above look. A strategy tiles a feature its own way, so the tiles
searched differ from one column to the next.

In [ ]:
"""Tabulate how many features of each class each strategy keeps."""

windows.classes(read)